In [1]:
import os
import shutil
from collections import defaultdict
from pathlib import Path
import random

random.seed(42)

base = r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset'
output = r'C:\Users\neo62\sperm-ai\data\processed\yolo_balanced'

# 출력 폴더 생성
for split in ['train', 'val']:
    Path(f'{output}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{output}/labels/{split}').mkdir(parents=True, exist_ok=True)

# train 데이터만 오버샘플링 (val은 그대로)
train_img = r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset\images\train'
train_lbl = r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset\labels\train'

# 각 파일에 어떤 클래스가 있는지 분류
sperm_files = []
cluster_files = []
small_files = []
mixed_files = []

for fname in os.listdir(train_lbl):
    if not fname.endswith('.txt'):
        continue
    fpath = os.path.join(train_lbl, fname)
    with open(fpath, 'r') as f:
        classes = set(int(line.split()[0]) for line in f if line.strip())
    
    if 1 in classes:  # cluster 포함
        cluster_files.append(fname)
    elif 2 in classes:  # small 포함
        small_files.append(fname)
    elif 0 in classes:  # sperm만
        sperm_files.append(fname)

print(f"sperm only 파일:   {len(sperm_files):,}개")
print(f"cluster 포함 파일: {len(cluster_files):,}개")
print(f"small 포함 파일:   {len(small_files):,}개")

# 목표: cluster 파일을 sperm 수준으로 오버샘플링
target = len(sperm_files)
cluster_oversampled = random.choices(cluster_files, k=target)
small_oversampled = random.choices(small_files, k=min(len(small_files)*3, target))

print(f"\n오버샘플링 후:")
print(f"sperm only:  {len(sperm_files):,}개")
print(f"cluster:     {len(cluster_oversampled):,}개")
print(f"small:       {len(small_oversampled):,}개")

# 파일 복사
def copy_set(file_list, split, prefix=''):
    for i, fname in enumerate(file_list):
        new_name = f"{prefix}{i:06d}_{fname}" if prefix else fname
        src_img = os.path.join(train_img, fname.replace('.txt', '.jpg'))
        src_lbl = os.path.join(train_lbl, fname)
        
        if os.path.exists(src_img) and os.path.exists(src_lbl):
            shutil.copy(src_img, f'{output}/images/{split}/{new_name.replace(".txt", ".jpg")}')
            shutil.copy(src_lbl, f'{output}/labels/{split}/{new_name}')

print("\n파일 복사 중...")
copy_set(sperm_files, 'train')
copy_set(cluster_oversampled, 'train', prefix='cls_')
copy_set(small_oversampled, 'train', prefix='sml_')

# val은 원본 그대로 복사
for fname in os.listdir(r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset\images\val'):
    shutil.copy(
        f'{base}/images/val/{fname}',
        f'{output}/images/val/{fname}'
    )
for fname in os.listdir(r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset\labels\val'):
    shutil.copy(
        f'{base}/labels/val/{fname}',
        f'{output}/labels/val/{fname}'
    )

total_train = len(os.listdir(f'{output}/images/train'))
total_val = len(os.listdir(f'{output}/images/val'))
print(f"\n✅ 완료!")
print(f"Train: {total_train:,}장")
print(f"Val:   {total_val:,}장")

sperm only 파일:   6,632개
cluster 포함 파일: 8,121개
small 포함 파일:   8,737개

오버샘플링 후:
sperm only:  6,632개
cluster:     6,632개
small:       6,632개

파일 복사 중...

✅ 완료!
Train: 19,896장
Val:   5,706장


In [2]:
yaml_content = """# VISEM-Tracking Balanced Dataset
path: C:/Users/neo62/sperm-ai/data/processed/yolo_balanced
train: images/train
val: images/val

nc: 3
names:
  0: sperm
  1: cluster
  2: small
"""

yaml_path = r'C:\Users\neo62\sperm-ai\data\processed\yolo_balanced\visem_balanced.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ visem_balanced.yaml 생성 완료")

✅ visem_balanced.yaml 생성 완료
